## Setup & Imports

# Image Preprocessing Pipeline for Diabetic Foot Ulcer Detection

## Complete Preprocessing Workflow (Chapter 3)

This notebook implements the full image preprocessing pipeline as described in the research methodology, following all steps outlined in Chapter 3.

## 1. Image Segmentation (HMRF-EM)

**Description:** Separate the foot from the background using Gaussian Mixture Model (GMM) with Hidden Markov Random Field (HMRF) and MAP (Maximum A Posteriori) estimation.

**Steps:**
- Convert RGB image to YDbDr color space
- Initialize 3 clusters (background, low-pressure, high-pressure) with K-means
- Run EM iterations with MAP labeling
- Apply segmentation mask to extract foot region

**Reference Code:** `preprocess_feet.py`

---

In [1]:
# Import Required Libraries
import os
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter, label, find_objects, binary_dilation
import cv2
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

# ============================================================
# 1. Color Space Conversion: RGB → YDbDr
# ============================================================

def rgb_to_ydbdr(image_rgb: np.ndarray) -> np.ndarray:
    """
    Convert an RGB image (H, W, 3) in [0, 255] to YDbDr.
    
    Y  =  0.299 R + 0.587 G + 0.114 B          (Luminance)
    Db = -0.450 R - 0.883 G + 1.333 B          (Chrominance blue)
    Dr = -1.333 R + 1.116 G + 0.217 B          (Chrominance red)
    
    Returns float64 array (H, W, 3).
    """
    img = image_rgb.astype(np.float64)
    R, G, B = img[..., 0], img[..., 1], img[..., 2]

    Y  =  0.299 * R + 0.587 * G + 0.114 * B
    Db = -0.450 * R - 0.883 * G + 1.333 * B
    Dr = -1.333 * R + 1.116 * G + 0.217 * B

    return np.stack([Y, Db, Dr], axis=-1)


# ============================================================
# 2. GMM Parameter Estimation (Expectation-Maximisation)
# ============================================================

class GaussianMixtureModel:
    """Diagonal-covariance GMM fitted with EM for D-dimensional data."""

    def __init__(self, K: int = 3, max_iter: int = 50, tol: float = 1e-4,
                 random_state: int = 42):
        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.rng = np.random.RandomState(random_state)
        self.weights = None   # (K,)
        self.means = None     # (K, D)
        self.covs = None      # (K, D, D)

    def _init_params(self, X: np.ndarray):
        N, D = X.shape
        indices = self.rng.choice(N, self.K, replace=False)
        self.means = X[indices].copy()
        self.covs = np.array([np.eye(D) * np.var(X, axis=0) for _ in range(self.K)])
        self.weights = np.ones(self.K) / self.K

    @staticmethod
    def _log_gaussian(X: np.ndarray, mu: np.ndarray, cov: np.ndarray) -> np.ndarray:
        """Log pdf of multivariate Gaussian (N,) for (N, D) data."""
        D = X.shape[1]
        diff = X - mu
        sign, log_det = np.linalg.slogdet(cov)
        inv_cov = np.linalg.inv(cov)
        mahal = np.sum(diff @ inv_cov * diff, axis=1)
        return -0.5 * (D * np.log(2 * np.pi) + log_det + mahal)

    def fit(self, X: np.ndarray):
        """Run EM on (N, D) data."""
        N, D = X.shape
        self._init_params(X)
        prev_ll = -np.inf

        for iteration in range(self.max_iter):
            # E-step
            log_resp = np.zeros((N, self.K))
            for k in range(self.K):
                log_resp[:, k] = np.log(self.weights[k] + 1e-300) + \
                                 self._log_gaussian(X, self.means[k], self.covs[k])
            # Log-sum-exp trick
            log_resp_max = log_resp.max(axis=1, keepdims=True)
            log_norm = log_resp_max + np.log(
                np.sum(np.exp(log_resp - log_resp_max), axis=1, keepdims=True))
            log_resp -= log_norm
            resp = np.exp(log_resp)

            # M-step
            Nk = resp.sum(axis=0)
            self.weights = Nk / N
            for k in range(self.K):
                self.means[k] = (resp[:, k:k+1].T @ X) / Nk[k]
                diff = X - self.means[k]
                self.covs[k] = (diff.T * resp[:, k]) @ diff / Nk[k]
                self.covs[k] += np.eye(D) * 1e-6

            ll = np.sum(log_norm)
            if abs(ll - prev_ll) < self.tol:
                print(f"  GMM converged at iteration {iteration + 1}")
                break
            prev_ll = ll

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return (N, K) responsibility matrix."""
        N = X.shape[0]
        log_resp = np.zeros((N, self.K))
        for k in range(self.K):
            log_resp[:, k] = np.log(self.weights[k] + 1e-300) + \
                             self._log_gaussian(X, self.means[k], self.covs[k])
        log_resp_max = log_resp.max(axis=1, keepdims=True)
        log_norm = log_resp_max + np.log(
            np.sum(np.exp(log_resp - log_resp_max), axis=1, keepdims=True))
        log_resp -= log_norm
        return np.exp(log_resp)

        # ============================================================
# 3. HMRF-MAP Segmentation
# ============================================================

def _neighbourhood_label_counts(labels: np.ndarray, K: int) -> np.ndarray:
    """For each pixel, count how many of its 4-connected neighbours share each label."""
    H, W = labels.shape
    counts = np.zeros((H, W, K), dtype=np.float64)
    for k in range(K):
        binary = (labels == k).astype(np.float64)
        count_k = np.zeros_like(binary)
        count_k[1:, :]  += binary[:-1, :]   # top
        count_k[:-1, :] += binary[1:, :]    # bottom
        count_k[:, 1:]  += binary[:, :-1]   # left
        count_k[:, :-1] += binary[:, 1:]    # right
        counts[..., k] = count_k
    return counts


def hmrf_em_segmentation(image_ydbdr: np.ndarray, K: int = 3,
                         beta: float = 1.5, max_iter: int = 30,
                         tol: float = 1e-3) -> np.ndarray:
    """HMRF-EM segmentation with GMM."""
    H, W, D = image_ydbdr.shape
    X = image_ydbdr.reshape(-1, D)
    N = X.shape[0]

    print("[1/3] Fitting GMM (K={}) on YDbDr features ...".format(K))
    gmm = GaussianMixtureModel(K=K, max_iter=80)
    gmm.fit(X)

    resp = gmm.predict_proba(X)
    labels = resp.argmax(axis=1).reshape(H, W)

    print("[2/3] Running HMRF-EM MAP Labeling (beta={}) ...".format(beta))

    likelihood_energy = np.zeros((N, K))
    for k in range(K):
        likelihood_energy[:, k] = -GaussianMixtureModel._log_gaussian(
            X, gmm.means[k], gmm.covs[k])
    likelihood_energy = likelihood_energy.reshape(H, W, K)

    for it in range(max_iter):
        nbr_counts = _neighbourhood_label_counts(labels, K)
        prior_energy = beta * (4.0 - nbr_counts)

        total_energy = likelihood_energy + prior_energy

        new_labels = total_energy.argmin(axis=2)

        changed = np.mean(new_labels != labels)
        labels = new_labels
        if changed < tol:
            print(f"  ICM converged at iteration {it + 1} (changed = {changed:.6f})")
            break

    print("[3/3] Re-estimating GMM on final labels ...")
    for k in range(K):
        mask_k = (labels.ravel() == k)
        if mask_k.sum() < D + 1:
            continue
        gmm.means[k] = X[mask_k].mean(axis=0)
        gmm.covs[k] = np.cov(X[mask_k].T) + np.eye(D) * 1e-6

    return labels


# ============================================================
# 4. Identify Foot Label & Create Mask
# ============================================================

def identify_foot_label(labels: np.ndarray, image_ydbdr: np.ndarray) -> int:
    """Identify which cluster corresponds to the foot contact region."""
    K = labels.max() + 1
    scores = np.zeros(K)
    for k in range(K):
        mask_k = (labels == k)
        if mask_k.sum() == 0:
            continue
        mean_y  = image_ydbdr[mask_k, 0].mean()
        mean_db = np.abs(image_ydbdr[mask_k, 1]).mean()
        mean_dr = np.abs(image_ydbdr[mask_k, 2]).mean()
        chrom_mag = np.sqrt(mean_db**2 + mean_dr**2)
        scores[k] = mean_y * (1.0 + chrom_mag)

    foot_label = int(np.argmax(scores))
    scores_dict = {i: round(float(s), 1) for i, s in enumerate(scores)}
    print(f"  Cluster scores: {scores_dict}")
    return foot_label


def create_foot_mask(labels: np.ndarray, foot_label: int) -> np.ndarray:
    """Binary mask: 1 = foot, 0 = background."""
    return np.where(labels == foot_label, 1, 0).astype(np.uint8)


def get_pure_sole_image(image_rgb: np.ndarray, foot_mask: np.ndarray,
               background: str = "black") -> np.ndarray:
    """Apply mask to image."""
    mask_3ch = foot_mask[:, :, np.newaxis]
    if background == "white":
        bg = np.full_like(image_rgb, 255)
        masked = image_rgb * mask_3ch + bg * (1 - mask_3ch)
    else:
        masked = image_rgb * mask_3ch
    return masked.astype(np.uint8)

✓ All libraries imported successfully


## 2. Morphological Operations with Dilation

**Description:** Apply dilation along the vertical axis to bridge gaps between disconnected regions (e.g., toes and foot).

**Steps:**
- Use dynamic kernel (approximately 12% of image height for vertical, 5px for horizontal)
- Apply binary dilation to connect toes with the main foot area
- Output: Extended segmentation mask

**Reference Code:** `split_feet.py` (lines 58-68)

---

In [2]:
# ============================================================
# SECTION 2: Morphological Operations with Dilation
# ============================================================

def apply_morphological_dilation(input_data, output_dir: str | None = None):
    """
    Apply morphological dilation to dilate foot region.
    
    Parameters
    ----------
    input_data : str or np.ndarray
        Path to segmented foot image OR segmented image array (H, W, 3)
    output_dir : str, optional
        Directory to save intermediate outputs (currently unused)
    
    Returns
    -------
    merged_mask : (H, W) binary array with dilated foot region
    image_array : (H, W, 3) original image array
    """
    print("=" * 60)
    print(f"  Morphological Dilation")
    print("=" * 60)

    # Load image if string, use directly if array
    if isinstance(input_data, str):
        img_pil = Image.open(input_data).convert("RGB")
        img_array = np.array(img_pil)
    else:
        img_array = input_data.copy()
    
    # Create binary mask from image
    intensity = img_array.sum(axis=2)
    binary_mask = (intensity > 0).astype(np.uint8)
    print(f"Binary mask created from image")
    
    # Dynamic kernel: 12% of image height for vertical, 5px for horizontal
    img_h, img_w = img_array.shape[:2]
    dilate_h = max(10, int(img_h * 0.12))
    struct_dilate = np.ones((dilate_h, 5), dtype=bool)
    print(f"Dilation kernel size: ({dilate_h}, 5)")
    
    # Apply binary dilation
    merged_mask = binary_dilation(binary_mask, structure=struct_dilate)
    print(f"Dilation applied - bridges gaps between disconnected regions (e.g., toes)")
    print("=" * 60)
    
    return merged_mask, img_array

## 3. Left-Right Separation & Square Crop

**Description:** Separate both footprints and create square images with consistent dimensions.

**Steps:**
- Apply Connected Component Analysis to find the 2 largest blobs
- Extract bounding boxes for left and right footprints
- Crop each foot tightly (from heel to toe)
- Pad shorter dimension with black pixels to create square images

**Reference Code:** `split_feet.py` (complete file)

---

In [3]:
# ============================================================
# SECTION 3: Left-Right Separation & Square Crop
# ============================================================

def pad_to_square(img_array: np.ndarray) -> np.ndarray:
    """Pad an image with black pixels to make it a perfect square."""
    h, w = img_array.shape[:2]
    size = max(h, w)
    
    pad_h_top = (size - h) // 2
    pad_h_bottom = size - h - pad_h_top
    pad_w_left = (size - w) // 2
    pad_w_right = size - w - pad_w_left
    
    padded = np.pad(img_array, ((pad_h_top, pad_h_bottom), 
                                (pad_w_left, pad_w_right), 
                                (0, 0)), 
                    mode='constant', constant_values=0)
    return padded


def separate_and_crop_feet(merged_mask: np.ndarray, img_array: np.ndarray, output_dir: str | None = None):
    """
    Separate left and right footprints and create square images.
    
    Parameters
    ----------
    merged_mask : (H, W) binary array with dilated foot region
    img_array : (H, W, 3) original image array
    output_dir : str, optional
        Directory to save intermediate visualizations
    
    Returns
    -------
    foot_left_img : (H, H, 3) left foot square image
    foot_right_img : (H, H, 3) right foot square image
    """
    print("=" * 60)
    print(f"  Left-Right Separation & Square Crop")
    print("=" * 60)
    
    # Connected Component Labeling (8-connected)
    structure = np.ones((3, 3), dtype=np.int32)
    labeled_mask, num_features = label(merged_mask, structure=structure)
    
    if num_features < 2:
        print("Error: Could not find at least 2 separate feet in the image.")
        return None, None
    
    print(f"Found {num_features} connected components")
    
    # Find the 2 Largest Components
    component_sizes = np.bincount(labeled_mask.ravel())
    component_sizes[0] = 0  # ignore background
    largest_labels = np.argsort(component_sizes)[-2:]
    print(f"2 largest components identified: labels {largest_labels}")
    
    # Extract Bounding Boxes
    slices = find_objects(labeled_mask)
    
    extracted_feet = []
    
    for lbl in largest_labels:
        bbox = slices[lbl - 1]
        y_slice, x_slice = bbox
        center_x = (x_slice.start + x_slice.stop) / 2.0
        
        # Tight crop
        cropped_img = img_array[y_slice, x_slice]
        
        # Pad to Square
        square_img = pad_to_square(cropped_img)
        
        extracted_feet.append({
            'center_x': center_x,
            'bbox': bbox,
            'square_img': square_img
        })
    
    # Sort by X coordinate (Leftmost = Left foot)
    extracted_feet.sort(key=lambda item: item['center_x'])
    foot_left_img  = extracted_feet[0]['square_img']
    foot_right_img = extracted_feet[1]['square_img']
    
    print(f"Left Foot:  {foot_left_img.shape[0]}x{foot_left_img.shape[1]}")
    print(f"Right Foot: {foot_right_img.shape[0]}x{foot_right_img.shape[1]}")
    print("=" * 60)
    
    return foot_left_img, foot_right_img

## 4. Grayscale Conversion & CLAHE Enhancement

**Description:** Convert to grayscale and enhance contrast using CLAHE to visualize pressure distribution better.

**Steps:**
- Convert RGB/segmented image to grayscale (focus on pressure intensity only)
- Apply Contrast Limited Adaptive Histogram Equalization (CLAHE)
  - Tile size: typically 8x8
  - Clip limit: prevents over-amplification of noise

**Formula:**
$$\text{Gray} = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$$

---

In [4]:
# ============================================================
# SECTION 4: Grayscale Conversion & CLAHE Enhancement
# ============================================================

def convert_to_grayscale(img_array: np.ndarray) -> np.ndarray:
    """
    Convert RGB image to grayscale using standard luminance formula.
    
    Formula: Gray = 0.299 * R + 0.587 * G + 0.114 * B
    
    Parameters
    ----------
    img_array : (H, W, 3) or (H, W) array
        Input RGB image or already grayscale image
    
    Returns
    -------
    gray_image : (H, W) uint8
        Grayscale image
    """
    if len(img_array.shape) == 3 and img_array.shape[2] == 3:
        # Using standard luminance formula for grayscale conversion
        gray = 0.299 * img_array[..., 0] + 0.587 * img_array[..., 1] + 0.114 * img_array[..., 2]
        return gray.astype(np.uint8)
    else:
        return img_array


def apply_clahe(image_gray: np.ndarray, clip_limit: float = 2.0, tile_grid_size: tuple = (8, 8)) -> np.ndarray:
    """
    Apply Contrast Limited Adaptive Histogram Equalization (CLAHE).
    
    CLAHE enhances local contrast by applying histogram equalization to small tiles
    of the image. This prevents noise amplification that occurs with standard histogram equalization.
    
    Parameters
    ----------
    image_gray : (H, W) uint8
        Grayscale image
    clip_limit : float
        Threshold for contrast limiting. Prevent over-amplification of noise.
        Default 2.0 is typical for pressure maps.
    tile_grid_size : tuple
        Size of local tiles for histogram equalization (height, width).
        Default (8, 8) is standard.
    
    Returns
    -------
    clahe_image : (H, W) uint8
        Enhanced grayscale image with improved contrast
    """
    # Create CLAHE object
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    
    # Apply CLAHE
    clahe_image = clahe.apply(image_gray)
    
    return clahe_image

## 5. Standardization and Normalization

### 5.1 Three-Channel RGB Conversion

**Description:** Convert grayscale image to 3-channel RGB for compatibility with pre-trained CNN models (ImageNet).

**Steps:**
- Copy grayscale values to all three channels: R = G = B
- Result: (H, W, 3) tensor

**Why:** Pre-trained models (EfficientNetB0, ResNet50, ConvNeXt-Tiny) expect 3-channel input

---

In [5]:
# ============================================================
# SECTION 5.1: Three-Channel RGB Conversion
# ============================================================

def convert_to_3channel_rgb(image_gray: np.ndarray) -> np.ndarray:
    """
    Convert grayscale image to 3-channel RGB by copying values to all channels.
    
    Parameters
    ----------
    image_gray : (H, W) uint8 or float grayscale image
    
    Returns
    -------
    image_rgb : (H, W, 3) 3-channel image
    """
    if len(image_gray.shape) == 2:
        # Stack grayscale 3 times to create RGB
        image_rgb = np.stack([image_gray, image_gray, image_gray], axis=-1)
        return image_rgb
    else:
        return image_gray

### 5.2 Image Resizing

**Description:** Resize all images to a uniform size (224×224 pixels) using bilinear interpolation.

**Specifications:**
- Target size: 224 × 224 pixels
- Interpolation method: Bilinear (preserves smooth transitions between pressure levels)
- Reason: Standard input size for EfficientNetB0, ResNet50, ConvNeXt-Tiny architectures

---

In [6]:
# ============================================================
# SECTION 5.2: Image Resizing
# ============================================================

def resize_image(image: np.ndarray, target_size: tuple = (224, 224), interpolation: str = 'bilinear') -> np.ndarray:
    """
    Resize image to target size using bilinear interpolation.
    
    Parameters
    ----------
    image : (H, W) or (H, W, C) array
    target_size : (height, width) tuple
    interpolation : 'bilinear' (default) or 'nearest'
    
    Returns
    -------
    resized_image : Array resized to target_size
    """
    pil_image = Image.fromarray(image)
    
    if interpolation == 'bilinear':
        interp = Image.BILINEAR
    else:
        interp = Image.NEAREST
    
    resized_pil = pil_image.resize((target_size[1], target_size[0]), interp)
    resized_array = np.array(resized_pil)
    
    return resized_array

### 5.3 Min-Max Normalization

**Description:** Normalize pixel values to the range [0, 1] for stable model training.

**Formula:**
$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

**Purpose:**
- Stabilizes gradient flow during backpropagation
- Ensures consistent input magnitude across images
- Improves model convergence

---

In [7]:
# ============================================================
# SECTION 5.3: Min-Max Normalization
# ============================================================

def min_max_normalize(image: np.ndarray) -> np.ndarray:
    """
    Normalize image pixel values to [0, 1] range using Min-Max scaling.
    
    Formula: x_norm = (x - x_min) / (x_max - x_min)
    
    Parameters
    ----------
    image : np.ndarray
        Input image (uint8 or float)
    
    Returns
    -------
    normalized_image : np.ndarray
        Normalized image with values in [0, 1]
    """
    image_float = image.astype(np.float32)
    x_min = image_float.min()
    x_max = image_float.max()
    
    if x_max == x_min:
        return np.zeros_like(image_float)
    
    normalized = (image_float - x_min) / (x_max - x_min)
    
    return normalized

## 6. Data Augmentation

**Description:** Apply random transformations to enhance model robustness and prevent overfitting.

**Augmentation Strategies:**
- **Random Rotations:** ±5 to ±10 degrees
  - Simulates natural variations in foot orientation
  - Accounts for camera angle variations during image capture

**Note:** Other techniques (flipping, warping) are NOT applied to maintain clinical relevance of pressure distribution.

---

In [8]:
# ============================================================
# SECTION 6: Data Augmentation (Random Rotations)
# ============================================================

def apply_random_rotation(image: np.ndarray, angle_range: tuple = (-10, 10), p: float = 0.5) -> np.ndarray:
    """
    Apply random rotation to image (±5 to ±10 degrees) using OpenCV.
    
    Parameters
    ----------
    image : (H, W, C) or (H, W) array
    angle_range : tuple (min_angle, max_angle) in degrees
    p : probability of applying augmentation
    
    Returns
    -------
    augmented_image : Rotated image
    """
    if np.random.rand() > p:
        return image
    
    angle = np.random.uniform(angle_range[0], angle_range[1])
    
    # Get image dimensions
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    
    # Get rotation matrix
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, scale=1.0)
    
    # Apply rotation with black border
    rotated = cv2.warpAffine(image, rotation_matrix, (w, h), 
                             borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    
    return rotated

## 7. Complete Pipeline Integration

**Description:** Create a unified preprocessing function that applies all steps in the correct order.

**Input:** Raw footprint image (RGB, variable size)

**Output:** Preprocessed image ready for CNN model training (3-channel, 224×224, normalized [0,1])

**Flow:**
1. Segmentation (HMRF-EM) → Mask
2. Apply Mask + Dilation → Connected Components
3. L-R Separation + Square Crop → Two square images
4. Grayscale + CLAHE Enhancement → Enhanced grayscale
5. 3-Channel Conversion → RGB (3-channel)
6. Resizing (224×224) → Standard size
7. Min-Max Normalization → [0, 1] range
8. Data Augmentation (optional, for training) → Augmented variants

---

In [9]:
# ============================================================
# SECTION 7: Complete Pipeline Integration
# ============================================================

def preprocess_foot_image(image_path: str, output_dir: str = None, augment: bool = False, target_size: tuple = (224, 224), save_final: bool = False):
    """
    Complete preprocessing pipeline for foot images.
    
    Input: Raw footprint image (RGB, variable size)
    Output: Preprocessed image ready for CNN (3-channel, 224×224, normalized [0,1])
    
    Flow:
    1. Segmentation (HMRF-EM) → Mask
    2. Apply Dilation → Extended segmentation
    3. L-R Separation + Square Crop → Two square images
    4. Grayscale + CLAHE → Enhanced grayscale
    5. 3-Channel Conversion → RGB
    6. Resizing (224×224) → Standard size
    7. Min-Max Normalization → [0, 1]
    8. Data Augmentation (optional) → Augmented variants
    
    Parameters
    ----------
    image_path : str
        Path to input foot image
    output_dir : str, optional
        Directory to save intermediate outputs
    augment : bool
        Whether to apply data augmentation
    target_size : tuple
        Target output size (height, width), default (224, 224)
    save_final : bool
        If True, save final preprocessed images as name_L.png and name_R.png
    
    Returns
    -------
    dict : {
        'left_foot': processed left foot image,
        'right_foot': processed right foot image,
        'left_foot_path': path to saved left foot (if save_final=True),
        'right_foot_path': path to saved right foot (if save_final=True)
    }
    """
    print("=" * 80)
    print(f"  COMPLETE IMAGE PREPROCESSING PIPELINE")
    print("=" * 80)
    
    # Step 1: Load and Segment
    print("\n[Step 1] Image Segmentation (HMRF-EM)...")
    img_pil = Image.open(image_path).convert("RGB")
    image_rgb = np.array(img_pil)
    
    image_ydbdr = rgb_to_ydbdr(image_rgb)
    labels = hmrf_em_segmentation(image_ydbdr, K=3, beta=1.5)
    foot_label = identify_foot_label(labels, image_ydbdr)
    foot_mask = create_foot_mask(labels, foot_label)
    pure_sole = get_pure_sole_image(image_rgb, foot_mask, background="black")
    
    # Step 2: Morphological Dilation
    print("\n[Step 2] Morphological Dilation...")
    merged_mask, _ = apply_morphological_dilation(pure_sole, output_dir=output_dir)
    
    # Step 3: L-R Separation & Square Crop
    print("\n[Step 3] L-R Separation & Square Crop...")
    foot_left, foot_right = separate_and_crop_feet(merged_mask, pure_sole, output_dir=output_dir)
    
    if foot_left is None or foot_right is None:
        print("Error: Failed to separate feet")
        return None
    
    # Step 4: Grayscale + CLAHE
    print("\n[Step 4] Grayscale & CLAHE Enhancement...")
    left_gray = convert_to_grayscale(foot_left)
    right_gray = convert_to_grayscale(foot_right)
    left_clahe = apply_clahe(left_gray, clip_limit=2.0, tile_grid_size=(8, 8))
    right_clahe = apply_clahe(right_gray, clip_limit=2.0, tile_grid_size=(8, 8))
    
    # Step 5: 3-Channel Conversion
    print("\n[Step 5] 3-Channel RGB Conversion...")
    left_rgb = convert_to_3channel_rgb(left_clahe)
    right_rgb = convert_to_3channel_rgb(right_clahe)
    
    # Step 6: Resizing
    print(f"\n[Step 6] Resizing to {target_size}...")
    left_resized = resize_image(left_rgb, target_size=target_size)
    right_resized = resize_image(right_rgb, target_size=target_size)
    
    # Step 7: Min-Max Normalization
    print("\n[Step 7] Min-Max Normalization [0, 1]...")
    left_normalized = min_max_normalize(left_resized)
    right_normalized = min_max_normalize(right_resized)
    
    result = {
        'left_foot': left_normalized,
        'right_foot': right_normalized,
    }
    
    # Step 8: Data Augmentation (optional)
    if augment:
        print("\n[Step 8] Data Augmentation (Random Rotation ±5-10°)...")
        left_aug = apply_random_rotation((left_normalized * 255).astype(np.uint8), angle_range=(-10, 10), p=1.0)
        right_aug = apply_random_rotation((right_normalized * 255).astype(np.uint8), angle_range=(-10, 10), p=1.0)
        result['left_foot_aug'] = min_max_normalize(left_aug)
        result['right_foot_aug'] = min_max_normalize(right_aug)
    
    # Save final outputs if requested
    if save_final:
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        
        # Create left and right folders
        left_output_dir = os.path.join(output_dir, "left_foot")
        right_output_dir = os.path.join(output_dir, "right_foot")
        os.makedirs(left_output_dir, exist_ok=True)
        os.makedirs(right_output_dir, exist_ok=True)
        
        # Save as uint8 (0-255) for storage and display
        left_output_path = os.path.join(left_output_dir, f"{base_name}_L.png")
        right_output_path = os.path.join(right_output_dir, f"{base_name}_R.png")
        
        Image.fromarray((left_normalized * 255).astype(np.uint8)).save(left_output_path)
        Image.fromarray((right_normalized * 255).astype(np.uint8)).save(right_output_path)
        
        result['left_foot_path'] = left_output_path
        result['right_foot_path'] = right_output_path
        
        print(f"\n✓ Saved final outputs:")
        print(f"  Left:  {left_output_path}")
        print(f"  Right: {right_output_path}")
    
    print("\n" + "=" * 80)
    print(f"  ✓ Preprocessing complete!")
    print(f"  Left foot shape: {result['left_foot'].shape}, range: [{result['left_foot'].min():.3f}, {result['left_foot'].max():.3f}]")
    print(f"  Right foot shape: {result['right_foot'].shape}, range: [{result['right_foot'].min():.3f}, {result['right_foot'].max():.3f}]")
    print("=" * 80)
    
    return result


def batch_preprocess_images(input_folder: str, output_folder: str, augment: bool = False):
    """
    Batch process all foot images in input folder.
    Saves final preprocessed images (left_foot and right_foot) to separate folders.
    
    Parameters
    ----------
    input_folder : str
        Folder containing raw foot images
    output_folder : str
        Base folder where output subfolders (left_foot, right_foot) will be created
    augment : bool
        Whether to apply data augmentation
    
    Returns
    -------
    results : dict
        Dictionary with image names as keys and preprocessing results as values
    """
    # Find all PNG images
    image_paths = sorted(glob.glob(os.path.join(input_folder, "*.png")))
    
    if len(image_paths) == 0:
        print(f"❌ No PNG images found in {input_folder}")
        return {}
    
    print(f"\n{'='*80}")
    print(f"  BATCH PREPROCESSING - REAL DATASET")
    print(f"{'='*80}")
    print(f"📁 Input folder:  {input_folder}")
    print(f"📂 Output folder: {output_folder}")
    print(f"📊 Found {len(image_paths)} images to process\n")
    
    results = {}
    successful = 0
    failed = 0
    
    for idx, img_path in enumerate(image_paths, 1):
        img_name = os.path.basename(img_path)
        print(f"\n[{idx}/{len(image_paths)}] Processing: {img_name}")
        print("-" * 80)
        
        try:
            result = preprocess_foot_image(
                img_path, 
                output_dir=output_folder, 
                augment=augment,
                save_final=True  # Save final normalized images
            )
            if result is not None:
                results[img_name] = result
                successful += 1
                print(f"✓ Successfully processed: {img_name}")
        except Exception as e:
            failed += 1
            print(f"✗ Error processing {img_name}:")
            print(f"  {str(e)}")
            import traceback
            traceback.print_exc()
            results[img_name] = None
    
    # Summary
    print("\n" + "=" * 80)
    print(f"  BATCH PROCESSING COMPLETE")
    print("=" * 80)
    print(f"✓ Successful: {successful}/{len(image_paths)}")
    print(f"✗ Failed:     {failed}/{len(image_paths)}")
    print(f"\n📂 Output structure:")
    print(f"  {output_folder}/")
    print(f"  ├── left_foot/  (contains name_L.png files)")
    print(f"  └── right_foot/ (contains name_R.png files)")
    print("=" * 80)
    
    return results

## Summary

This notebook provides a complete implementation of the image preprocessing pipeline as described in Chapter 3 of the research methodology. All 6 main preprocessing steps plus integration and testing are included.

### Key Outputs:
- ✅ Segmented foot regions
- ✅ Enhanced contrast images
- ✅ Standardized 224×224 RGB images
- ✅ Normalized pixel values [0, 1]
- ✅ Augmented training variants

## 9. Test Pipeline with P001.png

**Description:** Test the complete preprocessing pipeline with actual foot image and save outputs from each step.

**Output Location:** `E:\DFU\Image_PRE\` - Contains visualizations from all preprocessing steps.

---

In [10]:
# ============================================================
# TEST PIPELINE WITH P001.png - COMPLETE WORKFLOW
# ============================================================

# Path to test image and output directory
test_image_path = r"E:\DFU\Model\img\P001.png"
output_base_dir = r"E:\DFU\Image_PRE"

print("=" * 80)
print("  TESTING COMPLETE PREPROCESSING PIPELINE WITH P001.png")
print("=" * 80)
print(f"\nTest Image: {test_image_path}")
print(f"Output Directory: {output_base_dir}\n")

# ============== STEP 1: Image Segmentation (HMRF-EM) ==============
print("\n[STEP 1] Image Segmentation (HMRF-EM)...")
img_pil = Image.open(test_image_path).convert("RGB")
image_rgb = np.array(img_pil)
print(f"  Original image shape: {image_rgb.shape}")

# Convert to YDbDr
image_ydbdr = rgb_to_ydbdr(image_rgb)
print(f"  YDbDr conversion done")

# Segment
labels = hmrf_em_segmentation(image_ydbdr, K=3, beta=1.5)
foot_label = identify_foot_label(labels, image_ydbdr)
foot_mask = create_foot_mask(labels, foot_label)
pure_sole = get_pure_sole_image(image_rgb, foot_mask, background="black")

# Save segmentation results
os.makedirs(output_base_dir, exist_ok=True)
seg_output_path = os.path.join(output_base_dir, "01_segmentation_pure_sole.png")
Image.fromarray(pure_sole).save(seg_output_path)
print(f"  ✓ Saved: {seg_output_path}")

# ============== STEP 2: Morphological Dilation ==============
print("\n[STEP 2] Morphological Dilation...")
# Call apply_morphological_dilation function with pure_sole image
merged_mask, _ = apply_morphological_dilation(pure_sole, output_dir=output_base_dir)
dilation_output_path = os.path.join(output_base_dir, "02_morphological_dilation.png")
Image.fromarray((merged_mask * 255).astype(np.uint8)).save(dilation_output_path)
print(f"  ✓ Saved: {dilation_output_path}")

# ============== STEP 3: L-R Separation & Square Crop ==============
print("\n[STEP 3] Left-Right Separation & Square Crop...")
# Use pure_sole as the image array for foot extraction
foot_left, foot_right = separate_and_crop_feet(merged_mask, pure_sole, output_dir=output_base_dir)

if foot_left is not None and foot_right is not None:
    left_crop_path = os.path.join(output_base_dir, "03_left_foot_square_crop.png")
    right_crop_path = os.path.join(output_base_dir, "03_right_foot_square_crop.png")
    Image.fromarray(foot_left).save(left_crop_path)
    Image.fromarray(foot_right).save(right_crop_path)
    print(f"  ✓ Saved: {left_crop_path}")
    print(f"  ✓ Saved: {right_crop_path}")
else:
    print("  Error: Failed to separate feet. Skipping remaining steps.")
    raise ValueError("Failed to separate feet")

# ============== STEP 4: Grayscale & CLAHE Enhancement ==============
print("\n[STEP 4] Grayscale & CLAHE Enhancement...")
left_gray = convert_to_grayscale(foot_left)
right_gray = convert_to_grayscale(foot_right)

left_gray_path = os.path.join(output_base_dir, "04a_left_foot_grayscale.png")
right_gray_path = os.path.join(output_base_dir, "04a_right_foot_grayscale.png")
Image.fromarray(left_gray).save(left_gray_path)
Image.fromarray(right_gray).save(right_gray_path)
print(f"  ✓ Saved: {left_gray_path}")
print(f"  ✓ Saved: {right_gray_path}")

left_clahe = apply_clahe(left_gray, clip_limit=2.0, tile_grid_size=(8, 8))
right_clahe = apply_clahe(right_gray, clip_limit=2.0, tile_grid_size=(8, 8))

left_clahe_path = os.path.join(output_base_dir, "04b_left_foot_clahe.png")
right_clahe_path = os.path.join(output_base_dir, "04b_right_foot_clahe.png")
Image.fromarray(left_clahe).save(left_clahe_path)
Image.fromarray(right_clahe).save(right_clahe_path)
print(f"  ✓ Saved: {left_clahe_path}")
print(f"  ✓ Saved: {right_clahe_path}")

# ============== STEP 5: 3-Channel Conversion ==============
print("\n[STEP 5] 3-Channel RGB Conversion...")
left_rgb = convert_to_3channel_rgb(left_clahe)
right_rgb = convert_to_3channel_rgb(right_clahe)

left_rgb_path = os.path.join(output_base_dir, "05_left_foot_3channel_rgb.png")
right_rgb_path = os.path.join(output_base_dir, "05_right_foot_3channel_rgb.png")
Image.fromarray(left_rgb).save(left_rgb_path)
Image.fromarray(right_rgb).save(right_rgb_path)
print(f"  ✓ Saved: {left_rgb_path}")
print(f"  ✓ Saved: {right_rgb_path}")

# ============== STEP 6: Resizing (224x224) ==============
print("\n[STEP 6] Resizing to 224×224...")
target_size = (224, 224)
left_resized = resize_image(left_rgb, target_size=target_size)
right_resized = resize_image(right_rgb, target_size=target_size)

left_resized_path = os.path.join(output_base_dir, "06_left_foot_resized_224x224.png")
right_resized_path = os.path.join(output_base_dir, "06_right_foot_resized_224x224.png")
Image.fromarray(left_resized).save(left_resized_path)
Image.fromarray(right_resized).save(right_resized_path)
print(f"  Shape: {left_resized.shape}")
print(f"  ✓ Saved: {left_resized_path}")
print(f"  ✓ Saved: {right_resized_path}")

# ============== STEP 7: Min-Max Normalization ==============
print("\n[STEP 7] Min-Max Normalization [0, 1]...")
left_normalized = min_max_normalize(left_resized)
right_normalized = min_max_normalize(right_resized)

# Save normalized as display (scale to 0-255 for visualization)
left_norm_display = (left_normalized * 255).astype(np.uint8)
right_norm_display = (right_normalized * 255).astype(np.uint8)

left_norm_path = os.path.join(output_base_dir, "07_left_foot_normalized_display.png")
right_norm_path = os.path.join(output_base_dir, "07_right_foot_normalized_display.png")
Image.fromarray(left_norm_display).save(left_norm_path)
Image.fromarray(right_norm_display).save(right_norm_path)
print(f"  Left range: [{left_normalized.min():.4f}, {left_normalized.max():.4f}]")
print(f"  Right range: [{right_normalized.min():.4f}, {right_normalized.max():.4f}]")
print(f"  ✓ Saved: {left_norm_path}")
print(f"  ✓ Saved: {right_norm_path}")

# ============== STEP 8: Data Augmentation (optional) ==============
print("\n[STEP 8] Data Augmentation (Random Rotation ±5-10°)...")
np.random.seed(42)  # Fixed seed for reproducibility
left_aug = apply_random_rotation((left_normalized * 255).astype(np.uint8), angle_range=(-10, 10), p=1.0)
right_aug = apply_random_rotation((right_normalized * 255).astype(np.uint8), angle_range=(-10, 10), p=1.0)
left_aug_norm = min_max_normalize(left_aug)
right_aug_norm = min_max_normalize(right_aug)

left_aug_path = os.path.join(output_base_dir, "08_left_foot_augmented_rotation.png")
right_aug_path = os.path.join(output_base_dir, "08_right_foot_augmented_rotation.png")
Image.fromarray((left_aug_norm * 255).astype(np.uint8)).save(left_aug_path)
Image.fromarray((right_aug_norm * 255).astype(np.uint8)).save(right_aug_path)
print(f"  ✓ Saved: {left_aug_path}")
print(f"  ✓ Saved: {right_aug_path}")

print("\n" + "=" * 80)
print("  ✓ PIPELINE TEST COMPLETE!")
print("=" * 80)
print(f"\nAll outputs saved to: {output_base_dir}")
print("\nGenerated files:")
import glob
files = sorted(glob.glob(os.path.join(output_base_dir, "*.png")))
for i, f in enumerate(files, 1):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {i:2d}. {os.path.basename(f):45s} ({size_kb:7.1f} KB)")

  TESTING COMPLETE PREPROCESSING PIPELINE WITH P001.png

Test Image: E:\DFU\Model\img\P001.png
Output Directory: E:\DFU\Image_PRE


[STEP 1] Image Segmentation (HMRF-EM)...
  Original image shape: (331, 330, 3)
  YDbDr conversion done
[1/3] Fitting GMM (K=3) on YDbDr features ...
  GMM converged at iteration 35
[2/3] Running HMRF-EM MAP Labeling (beta=1.5) ...
  ICM converged at iteration 2 (changed = 0.000348)
[3/3] Re-estimating GMM on final labels ...
  Cluster scores: {0: 224.1, 1: 1817.5, 2: 17025.6}
  ✓ Saved: E:\DFU\Image_PRE\01_segmentation_pure_sole.png

[STEP 2] Morphological Dilation...
  Morphological Dilation
Binary mask created from image
Dilation kernel size: (39, 5)
Dilation applied - bridges gaps between disconnected regions (e.g., toes)
  ✓ Saved: E:\DFU\Image_PRE\02_morphological_dilation.png

[STEP 3] Left-Right Separation & Square Crop...
  Left-Right Separation & Square Crop
Found 2 connected components
2 largest components identified: labels [2 1]
Left Foot:  319

## 10. BATCH PROCESSING - Real Images from Dataset

**Description:** the complete preprocessing pipeline with actual foot image in dataset

**Output Location:** `E:\DFU\model\output` - Contains Left and Right with Image Preprocessing

---

In [11]:
# ============================================================
# BATCH PROCESSING - Real Images from Dataset
# ============================================================

# Define input and output directories
input_images_folder = r"E:\DFU\Model\img"
output_preprocessed_folder = r"E:\DFU\Model\img\output"

# Run batch processing
print("🚀 Starting batch preprocessing of diabetic foot images...\n")
batch_results = batch_preprocess_images(
    input_folder=input_images_folder,
    output_folder=output_preprocessed_folder,
    augment=False  # Set to True if you want augmented variants
)

# Print summary statistics
print(f"\n📊 FINAL SUMMARY")
print(f"Total images processed: {len(batch_results)}")
successful = sum(1 for v in batch_results.values() if v is not None)
print(f"✓ Successful: {successful}")
print(f"✗ Failed: {len(batch_results) - successful}")

if successful > 0:
    print(f"\n✨ All preprocessed images ready for model training!")
    print(f"📂 Location:")
    print(f"   Left feet:  {output_preprocessed_folder}/left_foot/")
    print(f"   Right feet: {output_preprocessed_folder}/right_foot/")

🚀 Starting batch preprocessing of diabetic foot images...


  BATCH PREPROCESSING - REAL DATASET
📁 Input folder:  E:\DFU\Model\img
📂 Output folder: E:\DFU\Model\img\output
📊 Found 1 images to process


[1/1] Processing: P001.png
--------------------------------------------------------------------------------
  COMPLETE IMAGE PREPROCESSING PIPELINE

[Step 1] Image Segmentation (HMRF-EM)...
[1/3] Fitting GMM (K=3) on YDbDr features ...
  GMM converged at iteration 35
[2/3] Running HMRF-EM MAP Labeling (beta=1.5) ...
  ICM converged at iteration 2 (changed = 0.000348)
[3/3] Re-estimating GMM on final labels ...
  Cluster scores: {0: 224.1, 1: 1817.5, 2: 17025.6}

[Step 2] Morphological Dilation...
  Morphological Dilation
Binary mask created from image
Dilation kernel size: (39, 5)
Dilation applied - bridges gaps between disconnected regions (e.g., toes)

[Step 3] L-R Separation & Square Crop...
  Left-Right Separation & Square Crop
Found 2 connected components
2 largest components identi

## 11. CLAHE Parameter Tuning

**Description:** Compare different CLAHE parameter settings to find the optimal contrast enhancement.

**Parameters tested:**
- clip_limit: [2.0, 3.0, 3.5, 4.0, 5.0]
- tile_size: [(8,8), (16,16), (4,4)]

**Output Location:** `E:\DFU\CLAHE_Tests\` - Contains comparison images with different settings.

---


In [12]:
# ============================================================
# CLAHE PARAMETER TUNING - Compare Different Settings
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

# Load test image
test_image_path = r"E:\DFU\Model\img\P001.png"
test_image = Image.open(test_image_path).convert('RGB')
test_array = np.array(test_image)

# Run full preprocessing to get segmented feet
print("Running preprocessing pipeline...")
image_ydbdr = rgb_to_ydbdr(test_array)
labels = hmrf_em_segmentation(image_ydbdr, K=3, beta=1.5)
foot_label = identify_foot_label(labels, image_ydbdr)
foot_mask = create_foot_mask(labels, foot_label)
pure_sole = get_pure_sole_image(test_array, foot_mask, background="black")

merged_mask, _ = apply_morphological_dilation(pure_sole)
foot_left, foot_right = separate_and_crop_feet(merged_mask, pure_sole)


# Convert to grayscale
left_gray = convert_to_grayscale(foot_left)
right_gray = convert_to_grayscale(foot_right)

# Test different CLAHE parameters
clahe_params = [
    {"clip_limit": 2.0, "tile_size": (8, 8)},
    {"clip_limit": 3.0, "tile_size": (8, 8)},
    {"clip_limit": 3.5, "tile_size": (8, 8)},
    {"clip_limit": 4.0, "tile_size": (8, 8)},
    {"clip_limit": 5.0, "tile_size": (8, 8)},
    {"clip_limit": 3.5, "tile_size": (16, 16)},
    {"clip_limit": 3.5, "tile_size": (4, 4)},
]

print("\nGenerating CLAHE variations for left foot...\n")
test_output_dir = Path(r"E:\DFU\CLAHE_Tests")
test_output_dir.mkdir(exist_ok=True)

for i, params in enumerate(clahe_params, 1):
    clip = params["clip_limit"]
    tile = params["tile_size"]
    
    # Apply CLAHE with current parameters
    left_enhanced = apply_clahe(left_gray, clip_limit=clip, tile_grid_size=tile)
    
    # Save as PNG
    filename = f"{i:02d}_left_clahe_clip{clip}_tile{tile[0]}.png"
    output_path = test_output_dir / filename
    Image.fromarray(left_enhanced).save(str(output_path))
    
    print(f"✓ {i}. clip_limit={clip}, tile_size={tile} → {filename}")

print(f"\n✨ All CLAHE variations saved to: {test_output_dir}")
print("\nCompare the images to choose the best setting!")


Running preprocessing pipeline...
[1/3] Fitting GMM (K=3) on YDbDr features ...
  GMM converged at iteration 35
[2/3] Running HMRF-EM MAP Labeling (beta=1.5) ...
  ICM converged at iteration 2 (changed = 0.000348)
[3/3] Re-estimating GMM on final labels ...
  Cluster scores: {0: 224.1, 1: 1817.5, 2: 17025.6}
  Morphological Dilation
Binary mask created from image
Dilation kernel size: (39, 5)
Dilation applied - bridges gaps between disconnected regions (e.g., toes)
  Left-Right Separation & Square Crop
Found 2 connected components
2 largest components identified: labels [2 1]
Left Foot:  319x319
Right Foot: 319x319

Generating CLAHE variations for left foot...

✓ 1. clip_limit=2.0, tile_size=(8, 8) → 01_left_clahe_clip2.0_tile8.png
✓ 2. clip_limit=3.0, tile_size=(8, 8) → 02_left_clahe_clip3.0_tile8.png
✓ 3. clip_limit=3.5, tile_size=(8, 8) → 03_left_clahe_clip3.5_tile8.png
✓ 4. clip_limit=4.0, tile_size=(8, 8) → 04_left_clahe_clip4.0_tile8.png
✓ 5. clip_limit=5.0, tile_size=(8, 8) → 05